# 張量、Dataset、DataLoader、nn.Module

對應原作業：Day 63–70、98。這是現代 PyTorch 練習；原始作業保留在 `../../homework/`。

從上到下執行即可。預設使用 CPU、合成資料，不需下載外部資料；改為真實資料時，先決定資料粒度與切分方式。

## 選擇架構
固定長度向量先用 MLP；影像看 CNN／預訓練模型；序列可試 GRU 或 Transformer。表格資料先與 scikit-learn 基準比較。`CrossEntropyLoss` 接受 logits 與 long 類別索引。

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
X = torch.randn(128, 4, dtype=torch.float32)
y = (X[:, 0] + X[:, 1] > 0).long()
loader = DataLoader(TensorDataset(X, y), batch_size=16, shuffle=True, num_workers=0)
xb, yb = next(iter(loader))
print(xb.shape, yb.shape, yb.dtype)


In [ ]:
class Classifier(nn.Module):
    def __init__(self, n_features: int, n_classes: int):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(n_features, 16), nn.ReLU(), nn.Linear(16, n_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

model = Classifier(n_features=4, n_classes=2)
logits = model(xb)
loss = nn.CrossEntropyLoss()(logits, yb)
assert logits.shape == (16, 2)
print("logits:", logits.shape, "loss:", loss.item())
